In [ ]:
!pip install nucleus-cdk==0.5.0rc2 | tail -n2

In [1]:
import pandas as pd
import numpy as np

# Export all GES's together

In [ ]:
## DATA WITHOUT
round_0_data = pd.read_csv("../raw-data/20260319-round-0-data-w-controls-w-reagents.csv")
round_0_ges = pd.read_csv("../raw-data/20260319-round-0-ges-obs.csv") # observed GES's round 0
round_0_ges_known = pd.read_csv('../raw-data/20260305-round-0-ges-known.csv') # manually annotated failures from round 0 GES's
round_1_ges = pd.read_csv("../raw-data/20260320-round-1-ges-obs.csv") # observed GES's round 1

In [5]:
round_1_ges.columns[~round_1_ges.columns.isin(round_0_ges.columns)]

Index([], dtype='object')

In [15]:
all_ges = pd.concat([round_0_ges, round_1_ges.sort_values('Name')])

In [16]:
all_ges.columns

Index(['experiment_id', 'Plate', 'Well', 'Read', 'Reader', 'Experiment',
       'Name', 'Type', '[DNA] (nM)', '[PEG4K 40%] (%)', '[RNAse Inhib] (U/mL)',
       '[PMix] (mg/mL)', '[Ribosome] (uM)', 'Rxn Volume (uL)',
       '[Magnesium acetate] (mM)', '[Creatine phosphate] (mM)',
       '[Potassium glutamate] (mM)', '[HEPES] (mM)', '[PPK] (uM)',
       '[PolyP] (mM)', '[ATP] (mM)', '[GTP] (mM)', '[CTP] (mM)', '[UTP] (mM)',
       '[TCEP] (mM)', '[Folinic acid] (mM)', '[Spermidine] (mM)',
       '[Amino acid mix] (mM)', '[tRNA] (ug/uL)', 'IsNEB', 'Product',
       'HasMgAR953', 'Date', 'Gain', 'Read Type',
       'sigmoid_steady_state (ng/uL)', 'sigmoid_rate (1/h)',
       'sigmoid_time_offset (h)', 'drift_rate (ng/uL/h)',
       'drift_rate_time_offset (h)', 'success'],
      dtype='object')

## Drop columns we don't need

In [27]:
all_ges_ed = all_ges.drop(columns=['Plate','Well', 'Experiment', 'Read', 'Name', 'Type', 'Date', 'Reader', 'Read Type', 'Gain']).fillna(0, axis=0).drop(columns=['drift_rate (ng/uL/h)', 'drift_rate_time_offset (h)', 'Product'])

In [33]:
all_ges_ed['isplamGFP'] = False
all_ges_ed['IsNEB'] = all_ges_ed['IsNEB'].astype(bool)
all_ges_ed['HasMgAR953'] = all_ges_ed['IsNEB'].astype(bool)

In [34]:
all_ges_ed

,experiment_id,[DNA] (nM),[PEG4K 40%] (%),[RNAse Inhib] (U/mL),[PMix] (mg/mL),[Ribosome] (uM),Rxn Volume (uL),[Magnesium acetate] (mM),[Creatine phosphate] (mM),[Potassium glutamate] (mM),...,[Spermidine] (mM),[Amino acid mix] (mM),[tRNA] (ug/uL),IsNEB,HasMgAR953,sigmoid_steady_state (ng/uL),sigmoid_rate (1/h),sigmoid_time_offset (h),success,isplamGFP
0,20260309-round-0-G8,2.996187,0.0,2000.000003,1.833326,1.8,10,12.469,0.0,100.0,...,2.0,0.3000,3.500,False,False,3.370488,4.565878,0.839765,True,False
1,20260309-round-0-G9,1.000000,0.0,2000.000003,1.555191,1.8,10,24.969,71.0,122.5,...,2.0,0.3000,3.500,False,False,12.284859,2.318723,1.254914,True,False
2,20260309-round-0-G10,2.996187,0.0,2000.000003,1.950000,1.8,10,24.969,100.0,60.0,...,2.0,0.3000,3.500,False,False,1.121951,3.976099,0.922998,True,False
3,20260309-round-0-G11,2.996187,0.0,2000.000003,1.950000,1.8,10,12.469,49.0,72.5,...,2.0,0.3000,3.500,False,False,1.598554,3.725096,0.922939,True,False
4,20260309-round-0-G12,2.996187,0.0,2000.000003,1.950000,1.8,10,14.969,20.0,140.0,...,2.0,0.3000,3.500,False,False,46.387345,3.044413,1.116157,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13,20260320-round-1-GES-31,5.000000,0.0,2000.000000,1.800000,1.8,10,5.000,31.0,40.0,...,2.0,0.2870,3.885,False,False,2.145794,5.675273,0.839666,False,False
12,20260320-round-1-GES-32,5.000000,0.0,2000.000000,1.800000,1.8,10,9.000,0.0,160.0,...,2.0,0.3115,1.925,False,False,12.396884,3.747402,0.838052,True,False
29,20260320-round-1-GES-33,5.000000,0.0,2000.000000,1.800000,1.8,10,12.000,4.0,112.5,...,2.0,0.3080,1.855,False,False,1.151286,3.253930,0.839322,True,False
4,20260320-round-1-GES-34,5.000000,0.0,2000.000000,1.800000,1.8,10,8.000,0.0,107.5,...,2.0,0.3990,3.255,False,False,8.539212,3.035050,1.087975,True,False


In [36]:
column_types = pd.DataFrame(data=np.reshape(["feature",]*len(all_ges_ed.columns), (1, len(all_ges_ed.columns))), columns=all_ges_ed.columns)
column_types.experiment_id.iloc[0] = 'id'
column_types.success.iloc[0] = 'classifier'
column_types.loc[0,'sigmoid_steady_state (ng/uL)':'sigmoid_time_offset (h)'] = 'regressor'

In [39]:
pd.concat([column_types, all_ges_ed],ignore_index=True)

,experiment_id,[DNA] (nM),[PEG4K 40%] (%),[RNAse Inhib] (U/mL),[PMix] (mg/mL),[Ribosome] (uM),Rxn Volume (uL),[Magnesium acetate] (mM),[Creatine phosphate] (mM),[Potassium glutamate] (mM),...,[Spermidine] (mM),[Amino acid mix] (mM),[tRNA] (ug/uL),IsNEB,HasMgAR953,sigmoid_steady_state (ng/uL),sigmoid_rate (1/h),sigmoid_time_offset (h),success,isplamGFP
0,id,feature,feature,feature,feature,feature,feature,feature,feature,feature,...,feature,feature,feature,feature,feature,regressor,regressor,regressor,classifier,feature
1,20260309-round-0-G8,2.996187,0.0,2000.000003,1.833326,1.8,10,12.469,0.0,100.0,...,2.0,0.3,3.5,False,False,3.370488,4.565878,0.839765,True,False
2,20260309-round-0-G9,1.0,0.0,2000.000003,1.555191,1.8,10,24.969,71.0,122.5,...,2.0,0.3,3.5,False,False,12.284859,2.318723,1.254914,True,False
3,20260309-round-0-G10,2.996187,0.0,2000.000003,1.95,1.8,10,24.969,100.0,60.0,...,2.0,0.3,3.5,False,False,1.121951,3.976099,0.922998,True,False
4,20260309-round-0-G11,2.996187,0.0,2000.000003,1.95,1.8,10,12.469,49.0,72.5,...,2.0,0.3,3.5,False,False,1.598554,3.725096,0.922939,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,20260320-round-1-GES-31,5.0,0.0,2000.0,1.8,1.8,10,5.0,31.0,40.0,...,2.0,0.287,3.885,False,False,2.145794,5.675273,0.839666,False,False
68,20260320-round-1-GES-32,5.0,0.0,2000.0,1.8,1.8,10,9.0,0.0,160.0,...,2.0,0.3115,1.925,False,False,12.396884,3.747402,0.838052,True,False
69,20260320-round-1-GES-33,5.0,0.0,2000.0,1.8,1.8,10,12.0,4.0,112.5,...,2.0,0.308,1.855,False,False,1.151286,3.25393,0.839322,True,False
70,20260320-round-1-GES-34,5.0,0.0,2000.0,1.8,1.8,10,8.0,0.0,107.5,...,2.0,0.399,3.255,False,False,8.539212,3.03505,1.087975,True,False


In [40]:
pd.concat([column_types, all_ges_ed],ignore_index=True).to_csv('20260324-round-1-all-ges-cumulative.csv', index=False)

# Combined dataset (w/o artifact IDs)

In [80]:
prev_dataset = pd.read_csv('../raw-data/20260309-round-1-data-allcols.csv',skiprows=[1])

In [81]:
prev_dataset['Experiment'] = prev_dataset.experiment_id.str.split('_').str[:-1].str.join('_')
prev_dataset['Experiment_Name'] = prev_dataset.experiment_id.str.split('_').str[-1]

In [82]:
prev_dataset['Experiment'].unique()

array(['20260309-round1', 'PMix-AR-898',
       'PMix_(AR-931)_QC_PURE_Timecourse',
       'PMix-MFG-128-PURE-Activity-QC', 'PPK_20250611', 'PPK_20250612',
       'PPK_20250613', 'PPK_20250616', 'Mg_K_screen-20251105_090558',
       'round0'], dtype=object)

In [83]:
prev_dataset.loc[prev_dataset.Experiment == '20260309-round1','Experiment'] = '20260309-round-0' 
prev_dataset.loc[prev_dataset.Experiment == 'round0','Experiment'] = '20260309-round-0-known'

In [85]:
prev_dataset.loc[:,'experiment_id'] = prev_dataset['Experiment'] + '-' + prev_dataset['Experiment_Name']

In [86]:
prev_dataset

,experiment_id,[DNA] (nM),[PEG4K 40%] (%),[RNAse Inhib] (U/mL),[PMix] (mg/mL),[Ribosome] (uM),Rxn Volume (uL),[Magnesium acetate] (mM),[Creatine phosphate] (mM),[Potassium glutamate] (mM),...,[tRNA] (ug/uL),IsNEB,HasMgAR953,sigmoid_steady_state (ng/uL),sigmoid_rate (1/h),sigmoid_time_offset (h),success,isplamGFP,Experiment,Experiment_Name
0,20260309-round-0-G8,2.996187,0.0,2000.000003,1.833326,1.8,10.0,12.469,0.000000,100.000000,...,3.5,False,True,3.370488,4.565878,0.839765,True,False,20260309-round-0,G8
1,20260309-round-0-G9,1.000000,0.0,2000.000003,1.555191,1.8,10.0,24.969,71.000000,122.500000,...,3.5,False,True,12.284859,2.318723,1.254914,True,False,20260309-round-0,G9
2,20260309-round-0-G10,2.996187,0.0,2000.000003,1.950000,1.8,10.0,24.969,100.000000,60.000000,...,3.5,False,True,1.121951,3.976099,0.922998,True,False,20260309-round-0,G10
3,20260309-round-0-G11,2.996187,0.0,2000.000003,1.950000,1.8,10.0,12.469,49.000000,72.500000,...,3.5,False,True,1.598554,3.725096,0.922939,True,False,20260309-round-0,G11
4,20260309-round-0-G12,2.996187,0.0,2000.000003,1.950000,1.8,10.0,14.969,20.000000,140.000000,...,3.5,False,True,46.387345,3.044413,1.116157,True,False,20260309-round-0,G12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,Mg_K_screen-20251105_090558-P3,5.000000,0.0,2000.000000,1.800000,1.8,5.0,7.000,20.000000,140.000000,...,3.5,True,False,44.246019,2.091836,1.455216,True,True,Mg_K_screen-20251105_090558,P3
178,Mg_K_screen-20251105_090558-P4,5.000000,0.0,2000.000000,1.800000,1.8,5.0,9.000,20.000000,100.000000,...,3.5,True,False,20.547366,2.272599,1.365400,True,True,Mg_K_screen-20251105_090558,P4
179,20260309-round-0-known-GES-9,19.999999,0.0,2000.000003,2.120920,1.8,10.0,0.000,0.000000,199.999997,...,NaN,False,False,0.000000,NaN,NaN,False,False,20260309-round-0-known,GES-9
180,20260309-round-0-known-GES-10,1.000000,0.0,2000.000003,1.682654,1.8,10.0,0.000,80.137303,190.847281,...,NaN,False,False,0.000000,NaN,NaN,False,False,20260309-round-0-known,GES-10


In [89]:
round_1_ges_ed = round_1_ges.sort_values('Name').drop(columns=['Plate','Well', 'Experiment', 'Read', 'Name', 'Type', 'Date', 'Reader', 'Read Type', 'Gain']).fillna(0, axis=0).drop(columns=['drift_rate (ng/uL/h)', 'drift_rate_time_offset (h)', 'Product'])
round_1_ges_ed['isplamGFP'] = False
round_1_ges_ed['IsNEB'] = False # round_1_ges_ed['IsNEB'].astype(bool)
round_1_ges_ed['HasMgAR953'] = False # round_1_ges_ed['IsNEB'].astype(bool)

In [95]:
round2_dataset = pd.concat([round_1_ges_ed, prev_dataset.drop(columns=['Experiment', 'Experiment_Name'])]).fillna(0)

In [96]:
pd.concat([column_types, round2_dataset],ignore_index=True).to_csv('20260324-round-2-data.csv',index=False)